# Notebook 4: Transformer Models (Trans, DSM, UMD, AMD)
**Dissertation:** Morphological Modeling of MSA and DA Using NLP with Deep Learning  
**Author:** Mohamed Medouani | Capitol Technology University

This notebook implements the four transformer-based architectures:
- **Trans:** Standard AraBERT/MARBERT fine-tuning — MSA WLA: **94.3%**
- **DSM:** Dialect-Specific Models — Best per-dialect performance
- **UMD:** Unified Multi-Dialect — Single model, all dialects
- **AMD:** Adaptive Multi-Dialect (LoRA) — **Best overall (DRS: 0.91)**

## Setup

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForTokenClassification, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'aubmindlab/bert-base-arabertv2'
NUM_LABELS = 15
print(f"Using device: {device}")
print(f"Base model: {MODEL_NAME}")

## 1. Standard Transformer (Trans)
**Pre-trained model:** AraBERT (aubmindlab/bert-base-arabertv2)  
**Context window:** 512 tokens  
**MSA WLA:** 94.3% (Table 12) — Highest among all models on MSA  
**Training:** 10 epochs, batch size 16, AdamW, weight decay 0.01

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer, TrainingArguments, Trainer

class TransformerMorphTagger:
    """
    Standard transformer fine-tuning for Arabic morphological tagging.
    Uses AraBERT (aubmindlab/bert-base-arabertv2) as the base model.
    Fine-tuned with task-specific token classification head.
    """
    def __init__(self, model_name=MODEL_NAME, num_labels=NUM_LABELS):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            hidden_dropout_prob=0.1,        # Dropout from Chapter 4
            attention_probs_dropout_prob=0.1
        )
        self.model.to(device)
        print(f"Transformer model loaded: {model_name}")
        print(f"Parameters: {sum(p.numel() for p in self.model.parameters()):,}")

    def get_training_args(self, output_dir='./trans_output'):
        """Training arguments matching Chapter 4 specifications."""
        return TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=10,            # From Chapter 4
            per_device_train_batch_size=16, # From Chapter 4
            per_device_eval_batch_size=32,
            warmup_ratio=0.1,               # Linear warmup
            weight_decay=0.01,              # AdamW weight decay
            learning_rate=2e-5,
            fp16=torch.cuda.is_available(), # Mixed precision (FP16) from Chapter 4
            evaluation_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_loss',
            report_to='none'
        )

# Demonstrate model loading
trans = TransformerMorphTagger()
print("\nModel config summary:")
print(f"  Hidden size: {trans.model.config.hidden_size}")
print(f"  Num attention heads: {trans.model.config.num_attention_heads}")
print(f"  Num hidden layers: {trans.model.config.num_hidden_layers}")
print(f"  Max position embeddings: {trans.model.config.max_position_embeddings}")

## 2. Dialect-Specific Models (DSM)
**Strategy:** Train a separate transformer for each dialect (MSA, EGY, LEV, GULF, MAGH)  
**Avg. DA WLA:** 82.4% (Table 20)  
**Advantage:** Highest per-dialect accuracy  
**Disadvantage:** 5x storage cost, requires dialect identification at inference

In [ ]:
class DialectSpecificModels:
    """
    Dialect-Specific Models (DSM): one fine-tuned transformer per Arabic variety.
    Each model is trained exclusively on its target dialect's data.
    At inference, dialect identification routes input to the correct model.
    """
    DIALECTS = ['MSA', 'EGY', 'LEV', 'GULF', 'MAGH']

    def __init__(self, model_name=MODEL_NAME, num_labels=NUM_LABELS):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.models = {}
        for dialect in self.DIALECTS:
            self.models[dialect] = AutoModelForTokenClassification.from_pretrained(
                model_name, num_labels=num_labels
            )
            print(f"  Loaded model for dialect: {dialect}")
        print(f"\nDSM: {len(self.models)} dialect-specific models initialized.")

    def identify_dialect(self, text):
        """
        Dialect identification (simplified placeholder).
        In production, this uses a separate dialect identification model.
        """
        # Simplified heuristic rules for demonstration
        if any(word in text for word in ['بحب', 'عايز', 'ازيك']):
            return 'EGY'
        elif any(word in text for word in ['شو', 'هيك', 'كيفك']):
            return 'LEV'
        elif any(word in text for word in ['وين', 'شلون', 'يبه']):
            return 'GULF'
        elif any(word in text for word in ['واش', 'ماشي', 'كاين']):
            return 'MAGH'
        else:
            return 'MSA'

    def predict(self, text):
        dialect = self.identify_dialect(text)
        model = self.models[dialect]
        print(f"Routing to {dialect} model for: '{text}'")
        return dialect, model

# Demonstration
dsm = DialectSpecificModels()
test_sentences = [
    "الرئيس يتحدث عن السياسة",
    "انا بحب القاهرة اوي",
    "شو بدك تاكل اليوم",
    "وين تروح هالحين",
    "واش كاين هنا"
]
for sent in test_sentences:
    dialect, _ = dsm.predict(sent)
    print(f"  '{sent}' → Routed to: {dialect}")

## 3. Unified Multi-Dialect (UMD) Model
**Strategy:** Single transformer trained on all dialects with dialect-embedding tokens  
**Avg. DA WLA:** 79.6% (Table 20)  
**DRS:** 0.79  
**Advantage:** Single model, handles code-switching well  
**Disadvantage:** Lower per-dialect accuracy than DSM

In [ ]:
class UnifiedMultiDialectModel(nn.Module):
    """
    Unified Multi-Dialect (UMD) model.
    A single transformer model trained on all Arabic varieties simultaneously.
    Dialect identity is injected via special dialect-identifier tokens prepended to input.
    """
    DIALECTS = ['MSA', 'EGY', 'LEV', 'GULF', 'MAGH']

    def __init__(self, model_name=MODEL_NAME, num_labels=NUM_LABELS):
        super(UnifiedMultiDialectModel, self).__init__()
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Add dialect tokens to vocabulary
        self.dialect_tokens = {d: f'[{d}]' for d in self.DIALECTS}
        new_tokens = list(self.dialect_tokens.values())
        self.tokenizer.add_special_tokens({'additional_special_tokens': new_tokens})
        
        # Load model and resize embeddings for new tokens
        self.model = AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=num_labels
        )
        self.model.resize_token_embeddings(len(self.tokenizer))
        
        print(f"UMD Model initialized.")
        print(f"  Vocabulary size (with dialect tokens): {len(self.tokenizer)}")
        print(f"  Dialect tokens: {new_tokens}")

    def prepare_input(self, text, dialect):
        """Prepend dialect token to input text."""
        dialect_token = self.dialect_tokens.get(dialect, '[MSA]')
        return f"{dialect_token} {text}"

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs

# Demonstration
umd = UnifiedMultiDialectModel()
sample_text = "الولد ذهب الى المدرسة"
prepared = umd.prepare_input(sample_text, 'MSA')
print(f"\nOriginal: {sample_text}")
print(f"With dialect token: {prepared}")

## 4. Adaptive Multi-Dialect (AMD) Model — The Winning Architecture
**Strategy:** Frozen base transformer + lightweight LoRA adapters per dialect  
**Avg. DA WLA:** 84.1% (Table 20) — Best among multi-dialect strategies  
**DRS:** 0.91 — Highest dialect robustness score  
**LoRA config:** rank=8, alpha=32, dropout=0.1  
**Key insight:** Transfer learning from MSA to low-resource dialects (especially Maghrebi)

In [ ]:
try:
    from peft import get_peft_model, LoraConfig, TaskType
    PEFT_AVAILABLE = True
except ImportError:
    PEFT_AVAILABLE = False
    print("PEFT not installed. Install with: pip install peft")

if PEFT_AVAILABLE:
    class AdaptiveMultiDialectModel(nn.Module):
        """
        Adaptive Multi-Dialect (AMD) model — the best-performing architecture.
        
        Key innovation: Uses Parameter-Efficient Fine-Tuning (PEFT) with LoRA adapters.
        - Base model (AraBERT) is pre-trained on MSA and FROZEN during dialect adaptation.
        - Lightweight LoRA adapters (rank=8) are inserted between transformer layers.
        - Each dialect gets its own set of adapters, trained independently.
        - At inference, the correct dialect's adapters are activated.
        
        Why this works for Arabic:
        - MSA is resource-rich; dialects are resource-poor.
        - Freezing the base model preserves MSA knowledge.
        - Small adapters (~0.5% of total params) capture dialect-specific patterns.
        - Reduces Dialect Transfer Gap by 34% vs standard fine-tuning.
        """
        def __init__(self, model_name=MODEL_NAME, num_labels=NUM_LABELS,
                     lora_rank=8, lora_alpha=32, lora_dropout=0.1):
            super(AdaptiveMultiDialectModel, self).__init__()
            
            # LoRA configuration (from Chapter 4)
            self.peft_config = LoraConfig(
                task_type=TaskType.TOKEN_CLS,
                inference_mode=False,
                r=lora_rank,            # Rank: 8 (from dissertation)
                lora_alpha=lora_alpha,  # Alpha: 32 (from dissertation)
                lora_dropout=lora_dropout,  # Dropout: 0.1 (from dissertation)
                target_modules=['query', 'value']  # Apply LoRA to attention layers
            )
            
            # Load base model and apply LoRA
            base = AutoModel.from_pretrained(model_name)
            self.encoder = get_peft_model(base, self.peft_config)
            
            # Classification head
            self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
            self.dropout = nn.Dropout(0.1)
            
            print("AMD Model initialized with LoRA adapters.")
            self.encoder.print_trainable_parameters()

        def forward(self, input_ids, attention_mask, labels=None):
            outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
            sequence_output = self.dropout(outputs.last_hidden_state)
            logits = self.classifier(sequence_output)
            
            loss = None
            if labels is not None:
                loss_fn = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)
                loss = loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
            
            return {'loss': loss, 'logits': logits}

    amd = AdaptiveMultiDialectModel()
    print("\nAMD model ready for dialect-specific fine-tuning.")
else:
    print("Install peft to use AMD: pip install peft")

## Architecture Comparison Summary

In [ ]:
import pandas as pd

comparison = {
    'Model': ['Trans (AraBERT)', 'DSM', 'UMD', 'AMD'],
    'MSA WLA (%)': [94.3, 94.3, 91.2, 93.8],
    'Avg DA WLA (%)': [78.4, 82.4, 79.6, 84.1],
    'DRS': [0.76, 0.84, 0.79, 0.91],
    'Num Models': [1, 5, 1, 1],
    'Trainable Params': ['~110M', '~110M x5', '~110M', '~0.5M (adapters)'],
    'Best For': ['MSA only', 'Per-dialect', 'Code-switching', 'All dialects']
}
df = pd.DataFrame(comparison)
print(df.to_string(index=False))
print("\n** AMD achieves the best balance of accuracy, robustness, and efficiency. **")